# Diabetic Retinopathy Grading — Production Pipeline v18## 30-Step Complete Build | Windows HP Victus (CPU+GPU) Optimised---

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# STEP 1 — System Check: GPU / CPU / Storage / CUDA / MPS / Memory
# ═══════════════════════════════════════════════════════════════════
import os, sys, platform, shutil, subprocess

print("=" * 65)
print("  SYSTEM DIAGNOSTICS — HP Victus Gaming Laptop")
print("=" * 65)
print(f"  OS           : {platform.system()} {platform.release()}")
print(f"  Python       : {sys.version.split()[0]}")
print(f"  Architecture : {platform.machine()}")
print(f"  CPU          : {platform.processor() or 'N/A'}")

# Memory
try:
    import psutil
    ram = psutil.virtual_memory()
    print(f"  RAM          : {ram.total / 1e9:.1f} GB (available: {ram.available / 1e9:.1f} GB)")
except ImportError:
    print("  RAM          : (install psutil for details)")

# Disk
total, used, free = shutil.disk_usage(os.path.expanduser("~"))
print(f"  Disk Free    : {free / 1e9:.1f} GB")

# GPU check (NVIDIA)
try:
    r = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                        "--format=csv,noheader"], capture_output=True, text=True, timeout=10)
    if r.returncode == 0:
        for line in r.stdout.strip().split("\n"):
            print(f"  GPU          : {line.strip()}")
    else:
        print("  GPU          : nvidia-smi failed — no NVIDIA GPU or driver issue")
except FileNotFoundError:
    print("  GPU          : nvidia-smi not found — CPU-only mode")

# CUDA quick check
try:
    import torch
    print(f"\n  PyTorch      : {torch.__version__}")
    print(f"  CUDA avail   : {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"  CUDA version : {torch.version.cuda}")
        print(f"  GPU (torch)  : {torch.cuda.get_device_name(0)}")
        print(f"  VRAM         : {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
except ImportError:
    print("  PyTorch      : NOT installed (will install in Step 2)")

print("=" * 65)
print("✅ System check complete.")


## Step 2 — Install Requirements

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# STEP 2 — Install / Upgrade All Dependencies
# ═══════════════════════════════════════════════════════════════════
import sys, subprocess

packages = [
    "torch>=2.1", "torchvision", "torchaudio",
    "timm>=1.0.0",
    "albumentations>=1.4.0",
    "opencv-python-headless",
    "scikit-learn",
    "scipy",
    "pandas",
    "numpy",
    "tqdm",
    "matplotlib",
    "grad-cam",
    "streamlit>=1.35.0",
    "huggingface_hub>=0.23.0",
    "kaggle",
    "pyarrow",
    "fastparquet",
    "pillow<11.0",
    "psutil",
]

print("🚀 Installing / upgrading dependencies...")
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--upgrade"] + packages,
    capture_output=True, text=True
)
if result.returncode == 0:
    print("✅ All dependencies installed successfully!")
else:
    print("⚠️ Some issues during installation:")
    print(result.stderr[-2000:])

print("🎯 Setup complete.")


## Step 3 — Imports, Device Detection & Path Setup

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# STEP 3 — All Imports, Device, Paths, Reproducibility, Resume Infra
# ═══════════════════════════════════════════════════════════════════
import os, sys, io, json, gc, time, random, shutil, warnings, pickle, platform
from pathlib import Path
from copy import deepcopy

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")  # safe for Windows; switch to inline for display
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import cv2
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_auc_score, cohen_kappa_score, ConfusionMatrixDisplay
)

warnings.filterwarnings("ignore")

# ── Reproducibility ───────────────────────────────────────────────
SEED = 42

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

seed_everything()

# ── Device (Windows HP Victus: NVIDIA GPU preferred) ──────────────
if torch.cuda.is_available():
    DEVICE = "cuda"
    print(f"🔥 CUDA GPU: {torch.cuda.get_device_name(0)}")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = "mps"
    print("🍎 Apple MPS detected")
else:
    DEVICE = "cpu"
    print("💻 Running on CPU")

USE_AMP = (DEVICE == "cuda")
print(f"\n✅ PyTorch {torch.__version__} | timm {timm.__version__}")
print(f"   Device: {DEVICE.upper()}  AMP: {'ON' if USE_AMP else 'OFF'}")

# ── Paths (Windows-safe) ─────────────────────────────────────────
DATA_DIR     = Path(os.environ.get("DATA_DIR",     str(Path.home() / "DR_data" / "aptos2019")))
ARTIFACT_DIR = Path(os.environ.get("ARTIFACT_DIR", str(Path.home() / "DR_data" / "artifacts_v18")))
IMG_DIR      = DATA_DIR / "train_images"
CSV_PATH     = DATA_DIR / "train.csv"
PLOT_DIR     = DATA_DIR / "plots_v18"

for d in [DATA_DIR, ARTIFACT_DIR, PLOT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Constants ─────────────────────────────────────────────────────
NUM_CLASSES  = 5
N_FOLDS      = 5
GRADE_MAP    = {0: "No DR", 1: "Mild DR", 2: "Moderate DR", 3: "Severe DR", 4: "Proliferative DR"}
GRADE_COLORS = ["#2ecc71", "#f1c40f", "#e67e22", "#e74c3c", "#8e44ad"]

# ── Safe torch.load (handles PyTorch 2.6+ weights_only) ──────────
def safe_load(path, map_location="cpu"):
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)

# ── Resume State (JSON-based, survives kernel restarts) ──────────
_STATE_FILE = ARTIFACT_DIR / "training_state.json"

def st_load():
    if _STATE_FILE.exists():
        return json.loads(_STATE_FILE.read_text())
    return {}

def st_save(key, value):
    s = st_load()
    s[key] = value
    _STATE_FILE.write_text(json.dumps(s, indent=2, default=str))

def st_get(key, default=None):
    return st_load().get(key, default)

print(f"\n📂 DATA_DIR     : {DATA_DIR}")
print(f"📂 ARTIFACT_DIR : {ARTIFACT_DIR}")
print(f"📂 PLOT_DIR     : {PLOT_DIR}")
print("✅ Step 3 complete — all imports & infra ready.")


## Step 4 — Kaggle Authentication

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# STEP 4 — Kaggle File Upload & Authentication
# ═══════════════════════════════════════════════════════════════════
KAGGLE_JSON = Path.home() / ".kaggle" / "kaggle.json"

if KAGGLE_JSON.exists():
    print(f"✅ kaggle.json found at {KAGGLE_JSON}")
else:
    ku = os.environ.get("KAGGLE_USERNAME", "")
    kk = os.environ.get("KAGGLE_KEY", "")
    if ku and kk:
        KAGGLE_JSON.parent.mkdir(exist_ok=True)
        KAGGLE_JSON.write_text(json.dumps({"username": ku, "key": kk}))
        if platform.system() != "Windows":
            KAGGLE_JSON.chmod(0o600)
        print("✅ kaggle.json written from environment variables.")
    else:
        print("⚠️  No Kaggle credentials found.")
        print("   Option 1: Set KAGGLE_USERNAME + KAGGLE_KEY env vars before running.")
        print("   Option 2: Place kaggle.json at:", KAGGLE_JSON)
        print("   Option 3: Manually download the dataset and place at:", DATA_DIR)
        print("\n   To get credentials: https://www.kaggle.com/settings → API → Create New Token")


## Step 5 — Dataset Download & Extraction (APTOS 2019)

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# STEP 5 — Download & Extract APTOS 2019 Blindness Detection Dataset
# ═══════════════════════════════════════════════════════════════════
import zipfile

_dl_flag = ARTIFACT_DIR / "_done_download.flag"

if (_dl_flag.exists()
        and IMG_DIR.exists()
        and CSV_PATH.exists()):
    n = len(list(IMG_DIR.glob("*.png")))
    print(f"✅ [RESUME] Dataset already present — {n:,} images.")
else:
    if not KAGGLE_JSON.exists():
        raise FileNotFoundError(
            "Kaggle credentials not configured. Run Step 4 first, "
            "or manually place the dataset at: " + str(DATA_DIR)
        )

    DATA_DIR.mkdir(parents=True, exist_ok=True)
    ZIP = DATA_DIR / "aptos2019.zip"

    if not ZIP.exists():
        print("Downloading APTOS 2019 dataset (~1.4 GB)...")
        r = subprocess.run(
            [sys.executable, "-m", "kaggle", "competitions", "download",
             "-c", "aptos2019-blindness-detection", "-p", str(DATA_DIR)],
            capture_output=True, text=True
        )
        if r.returncode != 0:
            print(r.stderr)
            raise RuntimeError("Kaggle download failed — check credentials / competition rules.")
        print("✅ Download complete.")
    else:
        print(f"✅ ZIP already present: {ZIP}")

    print("Extracting...")
    with zipfile.ZipFile(ZIP, "r") as zf:
        zf.extractall(DATA_DIR)
    print("✅ Extraction complete.")
    _dl_flag.touch()

n_imgs = len(list(IMG_DIR.glob("*.png"))) if IMG_DIR.exists() else 0
print(f"\n📂 Dataset ready — {n_imgs:,} training images at {IMG_DIR}")


## Step 6 — Load Dataset (train.csv + image paths + label mapping)

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# STEP 6 — Load CSV, Map Labels, Verify Image Paths
# ═══════════════════════════════════════════════════════════════════
if not CSV_PATH.exists():
    raise FileNotFoundError(f"train.csv not found at {CSV_PATH} — run Step 5 first.")

df = pd.read_csv(CSV_PATH)
df["image_path"]  = df["id_code"].apply(lambda x: str(IMG_DIR / f"{x}.png"))
df["grade_label"] = df["diagnosis"].map(GRADE_MAP)
df["binary"]      = (df["diagnosis"] >= 1).astype(int)

# Verify image files exist
missing = df[~df["image_path"].apply(lambda p: Path(p).exists())]
if len(missing):
    print(f"⚠️  {len(missing)} missing images — dropping them.")
    df = df[df["image_path"].apply(lambda p: Path(p).exists())].reset_index(drop=True)
else:
    print(f"✅ All {len(df):,} images found on disk.")

print(f"\nClass distribution:")
for g in range(5):
    n = (df["diagnosis"] == g).sum()
    bar = "█" * (n // 50)
    print(f"  Grade {g} ({GRADE_MAP[g]:20s}): {n:5d}  {bar}")

imb = df["diagnosis"].value_counts().max() / df["diagnosis"].value_counts().min()
print(f"\n  Total: {len(df):,}  |  Imbalance ratio: {imb:.1f}x")


## Step 7 — Data CleaningRemoves corrupted, black, severely blurry, and blue-artefact images.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# STEP 7 — Data Cleaning (Laplacian + Intensity + Blue-Channel Checks)
# ═══════════════════════════════════════════════════════════════════
_clean_flag = ARTIFACT_DIR / "_done_cleaning.flag"
_clean_path = ARTIFACT_DIR / "df_clean.parquet"

if _clean_flag.exists() and _clean_path.exists():
    df = pd.read_parquet(_clean_path)
    df["image_path"]  = df["id_code"].apply(lambda x: str(IMG_DIR / f"{x}.png"))
    df["grade_label"] = df["diagnosis"].map(GRADE_MAP)
    df["binary"]      = (df["diagnosis"] >= 1).astype(int)
    print(f"✅ [RESUME] Cleaned dataset loaded — {len(df):,} rows.")
else:
    print("Running data cleaning...")

    def _image_quality(path):
        bgr = cv2.imread(str(path))
        if bgr is None:
            return False, "unreadable"
        h, w = bgr.shape[:2]
        if h < 100 or w < 100:
            return False, "too_small"
        gray    = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
        lap_var = cv2.Laplacian(gray, cv2.CV_64F).var()
        bright  = float(gray.mean())
        if bright < 5.0:
            return False, "black_image"
        if lap_var < 30.0:
            return False, "blurry"
        b, g_ch, r = cv2.split(bgr)
        if float(b.mean()) > float(g_ch.mean()) * 1.8:
            return False, "blue_artefact"
        return True, "ok"

    results = []
    for _, row in tqdm(df.iterrows(), total=len(df), desc="Cleaning"):
        valid, reason = _image_quality(row["image_path"])
        results.append({"id_code": row["id_code"], "valid": valid, "reason": reason})

    res_df  = pd.DataFrame(results)
    invalid = res_df[~res_df["valid"]]
    print(f"  Total: {len(df):,}  |  Removed: {len(invalid):,}")
    for reason, cnt in invalid["reason"].value_counts().items():
        print(f"    {reason}: {cnt}")

    valid_ids = set(res_df[res_df["valid"]]["id_code"])
    df = df[df["id_code"].isin(valid_ids)].reset_index(drop=True)
    df["grade_label"] = df["diagnosis"].map(GRADE_MAP)
    df["binary"]      = (df["diagnosis"] >= 1).astype(int)
    df.to_parquet(_clean_path, index=False)
    _clean_flag.touch()
    print(f"✅ Clean dataset: {len(df):,} images saved.")

print("\nPost-cleaning class distribution:")
for g in range(5):
    n = (df["diagnosis"] == g).sum()
    print(f"  Grade {g} ({GRADE_MAP[g]:20s}): {n:5d}")


## Step 8 — Exploratory Data Analysis (EDA)

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# STEP 8 — EDA: Class Distribution, Sample Visualisation
# ═══════════════════════════════════════════════════════════════════
%matplotlib inline

_eda_flag = PLOT_DIR / "eda_distribution.png"

if _eda_flag.exists():
    print("✅ [RESUME] EDA plot already exists — displaying.")
    img_tmp = plt.imread(str(_eda_flag))
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.imshow(img_tmp); ax.axis("off")
    plt.tight_layout(); plt.show()
else:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    counts = df["diagnosis"].value_counts().sort_index()
    axes[0].bar([GRADE_MAP[i] for i in range(5)], counts.values,
                color=GRADE_COLORS, edgecolor="k", linewidth=0.6)
    axes[0].set_title("Class Distribution — APTOS 2019", fontweight="bold")
    axes[0].set_ylabel("Count")
    axes[0].tick_params(axis="x", rotation=20)
    for i, v in enumerate(counts.values):
        axes[0].text(i, v + 30, str(v), ha="center", fontsize=10)

    axes[1].pie(counts.values,
                labels=[GRADE_MAP[i] for i in range(5)],
                colors=GRADE_COLORS, autopct="%1.1f%%", startangle=90)
    axes[1].set_title("Class Proportions", fontweight="bold")

    plt.suptitle(f"APTOS 2019 — {len(df):,} images (post-cleaning)", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(_eda_flag, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"✅ EDA plot saved → {_eda_flag}")


## Step 9 — Label / Class Imbalance Analysis

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# STEP 9 — Label Analysis (class weights for loss + sampler)
# ═══════════════════════════════════════════════════════════════════
print("Label imbalance analysis:")
vc = df["diagnosis"].value_counts().sort_index()
class_counts = np.bincount(df["diagnosis"].values, minlength=NUM_CLASSES).astype(float)
class_weights_np = len(df) / (NUM_CLASSES * np.maximum(class_counts, 1))
class_weights_np = class_weights_np / class_weights_np.sum() * NUM_CLASSES

for g, c in vc.items():
    pct = c / len(df) * 100
    print(f"  Grade {g}: {c:5d} ({pct:5.1f}%)  weight = {class_weights_np[g]:.3f}")

print(f"\n  Imbalance ratio (max/min): {vc.max() / vc.min():.1f}x")
print("  → WeightedRandomSampler + class-weighted loss REQUIRED.")


## Step 10 — Preprocessing Pipeline (STRICT)Retina crop → Resize+Pad → Circular mask → CLAHE → Ben Graham → Green emphasis

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# STEP 10 — Fundus Preprocessing Functions
# ═══════════════════════════════════════════════════════════════════
IMG_SIZE = int(os.environ.get("IMG_SIZE", 512))
BG_SIGMA = max((IMG_SIZE // 10) | 1, 1)

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]


def _make_circular_mask(img):
    h, w = img.shape[:2]
    mask = np.zeros((h, w), np.uint8)
    cv2.circle(mask, (w // 2, h // 2), int(min(h, w) // 2 * 0.97), 255, -1)
    return mask


def _clahe_lab(rgb):
    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    lab[:, :, 0] = clahe.apply(lab[:, :, 0])
    return cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)


def _green_emphasis(rgb):
    out = rgb.copy().astype(np.float32)
    out[:, :, 1] = np.clip(out[:, :, 1] * 1.1, 0, 255)
    return out.astype(np.uint8)


def preprocess_fundus(path_or_array, size=None):
    """
    Full fundus preprocessing:
    1. Load RGB  2. Crop black borders  3. Resize+Pad to square
    4. Circular mask  5. CLAHE (LAB)  6. Ben Graham sharpening
    7. Green emphasis
    Returns: uint8 RGB [size × size × 3]
    """
    target = size or IMG_SIZE

    if isinstance(path_or_array, np.ndarray):
        rgb = path_or_array.copy()
    else:
        bgr = cv2.imread(str(path_or_array))
        if bgr is None:
            return np.zeros((target, target, 3), np.uint8)
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

    # Auto-crop black borders
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    _, thresh = cv2.threshold(gray, 10, 255, cv2.THRESH_BINARY)
    coords = cv2.findNonZero(thresh)
    if coords is not None:
        x, y, w, h = cv2.boundingRect(coords)
        rgb = rgb[y:y + h, x:x + w]

    # Resize keeping aspect ratio → pad to square
    h, w = rgb.shape[:2]
    scale = target / max(h, w)
    nh, nw = int(round(h * scale)), int(round(w * scale))
    rgb = cv2.resize(rgb, (nw, nh), interpolation=cv2.INTER_AREA)
    pt = (target - nh) // 2; pb = target - nh - pt
    pl = (target - nw) // 2; pr = target - nw - pl
    rgb = cv2.copyMakeBorder(rgb, pt, pb, pl, pr, cv2.BORDER_REFLECT_101)

    # Circular mask
    mask = _make_circular_mask(rgb)
    rgb[mask == 0] = 0

    # CLAHE in LAB space
    rgb = _clahe_lab(rgb)

    # Ben Graham: vessel sharpening
    sig = max((target // 10) | 1, 1)
    blur = cv2.GaussianBlur(rgb, (0, 0), sigmaX=sig)
    rgb = cv2.addWeighted(rgb, 4, blur, -4, 128)
    rgb[mask == 0] = 0

    # Green channel emphasis
    rgb = _green_emphasis(rgb)
    return rgb


print(f"✅ preprocess_fundus defined (IMG_SIZE={IMG_SIZE})")

# Quick benchmark
if len(df) > 0:
    t0 = time.time()
    for _ in range(3):
        preprocess_fundus(df["image_path"].iloc[0])
    lat = (time.time() - t0) / 3 * 1000
    print(f"   Avg latency: {lat:.1f} ms/image")


## Step 11 — Preprocessing CachePre-processes all images at 384px and saves as `.npy` to avoid recomputation.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# STEP 11 — Build Preprocessing Cache (384px .npy files)
# ═══════════════════════════════════════════════════════════════════
from concurrent.futures import ThreadPoolExecutor, as_completed

CACHE_DIR  = DATA_DIR / "cache_v18"
CACHE_FLAG = ARTIFACT_DIR / "_done_cache.flag"
CACHE_SIZE = 384

if CACHE_FLAG.exists() and CACHE_DIR.exists():
    n_cached = len(list(CACHE_DIR.glob("*.npy")))
    USE_CACHE = (n_cached >= len(df) * 0.95)
    print(f"✅ [RESUME] Cache exists — {n_cached:,} files at {CACHE_SIZE}px.  USE_CACHE={USE_CACHE}")
else:
    print(f"Building preprocessing cache at {CACHE_SIZE}px...")
    CACHE_DIR.mkdir(parents=True, exist_ok=True)

    def _cache_one(row):
        dst = CACHE_DIR / f'{row["id_code"]}.npy'
        if dst.exists():
            return True
        try:
            img = preprocess_fundus(row["image_path"], size=CACHE_SIZE)
            np.save(str(dst), img)
            return True
        except Exception:
            return False

    rows = [row for _, row in df.iterrows()]
    ok = fail = 0
    n_workers = min(4, os.cpu_count() or 1)
    with ThreadPoolExecutor(max_workers=n_workers) as ex:
        futs = {ex.submit(_cache_one, r): r["id_code"] for r in rows}
        for f in tqdm(as_completed(futs), total=len(futs), desc="Caching"):
            if f.result():
                ok += 1
            else:
                fail += 1

    print(f"✅ Cache done — {ok:,} ok, {fail} failed.")
    CACHE_FLAG.touch()
    USE_CACHE = True

print(f"\n   CACHE_DIR : {CACHE_DIR}")
print(f"   USE_CACHE : {USE_CACHE}")


## Step 12 — Train / Test Split (Hold-Out)Fold 0 = held-out test set, never used during training.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# STEP 12 — Hold-Out Test Split (fold 0 reserved)
# ═══════════════════════════════════════════════════════════════════
# Note: The actual fold assignment happens in Step 13 (StratifiedKFold).
# Fold 0 is designated as the hold-out test set.
# This cell just documents the strategy.

print("Hold-out test strategy:")
print("  • Fold 0 = HELD-OUT TEST SET (never seen during training)")
print("  • Folds 1-4 = used for cross-validation training")
print("  • This prevents any data leakage between training and evaluation.")
print("\n  → K-Fold split is created in Step 13.")


## Step 13 — Stratified K-Fold Split (5 folds)

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# STEP 13 — StratifiedKFold = 5, preserving class distribution
# ═══════════════════════════════════════════════════════════════════
_splits_path = ARTIFACT_DIR / "kfold_splits.parquet"

if _splits_path.exists():
    df = pd.read_parquet(_splits_path)
    df["image_path"]  = df["id_code"].apply(lambda x: str(IMG_DIR / f"{x}.png"))
    df["grade_label"] = df["diagnosis"].map(GRADE_MAP)
    df["binary"]      = (df["diagnosis"] >= 1).astype(int)
    print(f"✅ [RESUME] K-Fold splits loaded.")
else:
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    df["fold"] = -1
    for fold_idx, (_, val_idx) in enumerate(skf.split(df, df["diagnosis"])):
        df.loc[val_idx, "fold"] = fold_idx
    df.to_parquet(_splits_path, index=False)
    print(f"✅ 5-Fold splits created and saved.")

print(f"\nFold distribution:")
for fold in range(N_FOLDS):
    n  = (df["fold"] == fold).sum()
    gd = df[df["fold"] == fold]["diagnosis"].value_counts().sort_index()
    gs = " | ".join([f"G{g}:{c}" for g, c in gd.items()])
    print(f"  Fold {fold}: {n:5d} samples  [{gs}]")


## Step 14 — Augmentation Pipelines + Dataset Classes + WeightedRandomSampler

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# STEP 14 — Augmentations, Dataset Classes, Sampler, Loader Factory
# ═══════════════════════════════════════════════════════════════════

# ── Augmentation Pipelines ────────────────────────────────────────
def build_train_transforms(img_size=IMG_SIZE):
    return A.Compose([
        A.RandomResizedCrop(height=img_size, width=img_size,
                            scale=(0.7, 1.0), ratio=(0.9, 1.1), p=1.0),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.Rotate(limit=15, p=0.7),
        A.CLAHE(clip_limit=2.0, tile_grid_size=(8, 8), p=0.3),
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
        A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=10, p=0.3),
        A.RandomGamma(gamma_limit=(80, 120), p=0.3),
        A.GaussNoise(var_limit=(10.0, 50.0), p=0.2),
        A.MotionBlur(blur_limit=3, p=0.1),
        A.CoarseDropout(max_holes=8,
                        max_height=img_size // 16, max_width=img_size // 16,
                        min_holes=1, p=0.2),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])

def build_val_transforms(img_size=IMG_SIZE):
    return A.Compose([
        A.Resize(img_size, img_size),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])

def build_tta_transforms(img_size=IMG_SIZE):
    """5-view TTA: original + HFlip + Rotate±10 + VFlip."""
    n = A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
    r = A.Resize(img_size, img_size)
    return [
        build_val_transforms(img_size),
        A.Compose([r, A.HorizontalFlip(p=1.0), n, ToTensorV2()]),
        A.Compose([r, A.Rotate(limit=(10, 10), p=1.0), n, ToTensorV2()]),
        A.Compose([r, A.Rotate(limit=(-10, -10), p=1.0), n, ToTensorV2()]),
        A.Compose([r, A.VerticalFlip(p=1.0), n, ToTensorV2()]),
    ]

# ── Dataset Classes ───────────────────────────────────────────────
class APTOSDataset(Dataset):
    """5-class grade dataset with cache support."""
    def __init__(self, df, transform=None, img_size=None, use_cache=True):
        self.df        = df.reset_index(drop=True)
        self.transform = transform
        self.img_size  = img_size or IMG_SIZE
        self.use_cache = use_cache and USE_CACHE

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        label = int(row["diagnosis"])

        if self.use_cache:
            cache_path = CACHE_DIR / f'{row["id_code"]}.npy'
            if cache_path.exists():
                img = np.load(str(cache_path))
                if self.img_size != CACHE_SIZE:
                    img = cv2.resize(img, (self.img_size, self.img_size),
                                     interpolation=cv2.INTER_AREA)
            else:
                img = preprocess_fundus(row["image_path"], size=self.img_size)
        else:
            img = preprocess_fundus(row["image_path"], size=self.img_size)

        if self.transform:
            img = self.transform(image=img)["image"]
        else:
            img = torch.from_numpy(img.transpose(2, 0, 1)).float() / 255.0
        return img, label


class BinaryAPTOSDataset(Dataset):
    """Stage-1 binary: 0=No DR, 1=any DR."""
    def __init__(self, df, transform=None, img_size=None, use_cache=True):
        self.df        = df.reset_index(drop=True)
        self.transform = transform
        self.img_size  = img_size or IMG_SIZE
        self.use_cache = use_cache and USE_CACHE

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        label = torch.tensor(float(row["diagnosis"] >= 1), dtype=torch.float)
        if self.use_cache:
            cp = CACHE_DIR / f'{row["id_code"]}.npy'
            img = np.load(str(cp)) if cp.exists() else preprocess_fundus(row["image_path"], size=self.img_size)
            if cp.exists() and self.img_size != CACHE_SIZE:
                img = cv2.resize(img, (self.img_size, self.img_size), interpolation=cv2.INTER_AREA)
        else:
            img = preprocess_fundus(row["image_path"], size=self.img_size)
        if self.transform:
            img = self.transform(image=img)["image"]
        return img, label


class OrdinalAPTOSDataset(Dataset):
    """Stage-2 ordinal (DR-positive only, grades 1-4)."""
    ORDINAL = {1: [1,0,0,0], 2: [1,1,0,0], 3: [1,1,1,0], 4: [1,1,1,1]}

    def __init__(self, df, transform=None, img_size=None, use_cache=True):
        self.df        = df[df["diagnosis"] >= 1].reset_index(drop=True)
        self.transform = transform
        self.img_size  = img_size or IMG_SIZE
        self.use_cache = use_cache and USE_CACHE

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row   = self.df.iloc[idx]
        grade = int(row["diagnosis"])
        label = torch.tensor(self.ORDINAL[grade], dtype=torch.float)
        if self.use_cache:
            cp = CACHE_DIR / f'{row["id_code"]}.npy'
            img = np.load(str(cp)) if cp.exists() else preprocess_fundus(row["image_path"], size=self.img_size)
            if cp.exists() and self.img_size != CACHE_SIZE:
                img = cv2.resize(img, (self.img_size, self.img_size), interpolation=cv2.INTER_AREA)
        else:
            img = preprocess_fundus(row["image_path"], size=self.img_size)
        if self.transform:
            img = self.transform(image=img)["image"]
        return img, label


# ── WeightedRandomSampler ────────────────────────────────────────
def build_weighted_sampler(df_split):
    labels = df_split["diagnosis"].values
    counts = np.bincount(labels, minlength=NUM_CLASSES).astype(float)
    w_per_class = 1.0 / np.maximum(counts, 1)
    sample_weights = torch.tensor([w_per_class[l] for l in labels], dtype=torch.double)
    return WeightedRandomSampler(sample_weights, num_samples=len(sample_weights), replacement=True)


# ── DataLoader Factory ───────────────────────────────────────────
def make_loader(dataset, batch_size=16, shuffle=False, sampler=None, drop_last=False):
    nw = min(4, os.cpu_count() or 1)
    # Windows: num_workers > 0 can cause issues; use 0 if problems arise
    if platform.system() == "Windows":
        nw = 0
    return DataLoader(
        dataset, batch_size=batch_size, shuffle=(shuffle and sampler is None),
        sampler=sampler, num_workers=nw, pin_memory=(DEVICE == "cuda"),
        drop_last=drop_last, persistent_workers=(nw > 0)
    )


train_transforms   = build_train_transforms(IMG_SIZE)
val_transforms     = build_val_transforms(IMG_SIZE)
tta_transforms_lst = build_tta_transforms(IMG_SIZE)

print(f"✅ Augmentation pipelines built (train ops: {len(train_transforms.transforms)}, TTA views: {len(tta_transforms_lst)})")
print(f"✅ Dataset classes: APTOSDataset, BinaryAPTOSDataset, OrdinalAPTOSDataset")
print(f"✅ WeightedRandomSampler + DataLoader factory ready")
print(f"   num_workers = {'0 (Windows safe)' if platform.system() == 'Windows' else min(4, os.cpu_count() or 1)}")


## Step 15 — DataLoader Creation (train / val / test)

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# STEP 15 — Create DataLoaders for a Quick Sanity Check
# ═══════════════════════════════════════════════════════════════════
# Batch sizes tuned for HP Victus (~4-8GB VRAM)
BS_MAP = {224: 32, 384: 16, 512: 8}

# Quick sanity check with fold 0 as val
df_train_tmp = df[df["fold"] != 0].reset_index(drop=True)
df_val_tmp   = df[df["fold"] == 0].reset_index(drop=True)

_bs = BS_MAP.get(IMG_SIZE, 16)
_tr_ds = APTOSDataset(df_train_tmp, transform=build_val_transforms(224), img_size=224)
_va_ds = APTOSDataset(df_val_tmp,   transform=build_val_transforms(224), img_size=224)
_tr_ld = make_loader(_tr_ds, batch_size=_bs, shuffle=True)
_va_ld = make_loader(_va_ds, batch_size=_bs, shuffle=False)

# Fetch one batch to verify
imgs, labels = next(iter(_tr_ld))
print(f"✅ DataLoader sanity check:")
print(f"   Train batches : {len(_tr_ld)}")
print(f"   Val batches   : {len(_va_ld)}")
print(f"   Batch shape   : {imgs.shape}")
print(f"   Labels        : {labels[:8].tolist()}")
print(f"   Batch sizes   : {BS_MAP}")

del _tr_ds, _va_ds, _tr_ld, _va_ld, imgs, labels
gc.collect()


## Step 16 — Model Initialisation (EfficientNetV2 + GeM + Custom Head)

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# STEP 16 — Model Architecture: EfficientNetV2-S + GeM + Hybrid Head
# ═══════════════════════════════════════════════════════════════════
BACKBONE = os.environ.get("BACKBONE", "tf_efficientnetv2_s")


class GeM(nn.Module):
    """Generalized Mean Pooling."""
    def __init__(self, p=3, eps=1e-6):
        super().__init__()
        self.p   = nn.Parameter(torch.ones(1) * p)
        self.eps = eps

    def forward(self, x):
        return F.avg_pool2d(
            x.clamp(min=self.eps).pow(self.p),
            (x.size(-2), x.size(-1))
        ).pow(1.0 / self.p)


class DRModel(nn.Module):
    """
    DR model with configurable head.
    mode='softmax' → 5-class | mode='sigmoid' → binary/ordinal
    """
    def __init__(self, backbone=BACKBONE, num_classes=5,
                 dropout=0.4, pretrained=True, mode="softmax"):
        super().__init__()
        self.mode = mode
        self.backbone = timm.create_model(
            backbone, pretrained=pretrained,
            features_only=False, num_classes=0, global_pool=""
        )
        feat_dim  = self.backbone.num_features
        self.pool = GeM(p=3)
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.BatchNorm1d(feat_dim),
            nn.Linear(feat_dim, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        feat   = self.backbone(x)
        pooled = self.pool(feat)
        logits = self.head(pooled)
        return logits

    def freeze_backbone(self):
        for p in self.backbone.parameters():
            p.requires_grad_(False)

    def unfreeze_top_blocks(self, n=4):
        for p in self.backbone.parameters():
            p.requires_grad_(False)
        blocks = list(self.backbone.blocks)
        for block in blocks[-n:]:
            for p in block.parameters():
                p.requires_grad_(True)
        for attr in ["conv_head", "bn2", "norm_head"]:
            if hasattr(self.backbone, attr):
                for p in getattr(self.backbone, attr).parameters():
                    p.requires_grad_(True)

    def unfreeze_all(self):
        for p in self.parameters():
            p.requires_grad_(True)


def build_5class_model(pretrained=True):
    return DRModel(BACKBONE, num_classes=5, dropout=0.4,
                   pretrained=pretrained, mode="softmax").to(DEVICE)

def build_stage1_model(pretrained=True):
    return DRModel(BACKBONE, num_classes=1, dropout=0.4,
                   pretrained=pretrained, mode="sigmoid").to(DEVICE)

def build_stage2_model(pretrained=True):
    return DRModel(BACKBONE, num_classes=4, dropout=0.4,
                   pretrained=pretrained, mode="sigmoid").to(DEVICE)


# Quick parameter count
_tmp = build_5class_model(pretrained=False)
total_params = sum(p.numel() for p in _tmp.parameters())
print(f"✅ DRModel defined — backbone: {BACKBONE}")
print(f"   Total params: {total_params / 1e6:.2f}M")
del _tmp; gc.collect()
if DEVICE == "cuda":
    torch.cuda.empty_cache()


## Step 17 — Loss Functions + Optimizer + Scheduler

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# STEP 17 — Loss Functions, MixUp, Metrics, Criterion Builder
# ═══════════════════════════════════════════════════════════════════

class FocalLoss(nn.Module):
    """Multi-class Focal Loss with optional class weights."""
    def __init__(self, alpha=None, gamma=2.0, reduction="mean"):
        super().__init__()
        self.alpha, self.gamma, self.reduction = alpha, gamma, reduction

    def forward(self, inputs, targets):
        ce   = F.cross_entropy(inputs, targets, weight=self.alpha, reduction="none")
        pt   = torch.exp(-ce)
        loss = ((1 - pt) ** self.gamma) * ce
        return loss.mean() if self.reduction == "mean" else loss.sum()


class BinaryFocalLoss(nn.Module):
    """Binary Focal Loss for Stage-1."""
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha, self.gamma = alpha, gamma

    def forward(self, logits, targets):
        bce  = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        prob = torch.sigmoid(logits)
        p_t  = prob * targets + (1 - prob) * (1 - targets)
        a_t  = self.alpha * targets + (1 - self.alpha) * (1 - targets)
        return (a_t * (1 - p_t) ** self.gamma * bce).mean()


def compute_class_weights(labels, num_classes=5):
    counts  = np.bincount(labels, minlength=num_classes).astype(float)
    weights = len(labels) / (num_classes * np.maximum(counts, 1))
    weights = weights / weights.sum() * num_classes
    return torch.tensor(weights, dtype=torch.float)


def mixup_data(x, y, alpha=0.4, device=DEVICE):
    lam = max(np.random.beta(alpha, alpha), 1 - np.random.beta(alpha, alpha)) if alpha > 0 else 1.0
    idx = torch.randperm(x.size(0)).to(device)
    return lam * x + (1 - lam) * x[idx], y, y[idx], lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)


def qwk(y_true, y_pred):
    return cohen_kappa_score(y_true, y_pred, weights="quadratic")

def accuracy(y_true, y_pred):
    return (np.array(y_true) == np.array(y_pred)).mean()


def build_5class_criterion(df_split, device=DEVICE):
    """Hybrid 0.5×WCE + 0.5×Focal with label smoothing."""
    cw = compute_class_weights(df_split["diagnosis"].values).to(device)
    ce = nn.CrossEntropyLoss(weight=cw, label_smoothing=0.1)
    fl = FocalLoss(alpha=cw, gamma=2.0)
    def criterion(logits, labels):
        return 0.5 * ce(logits, labels) + 0.5 * fl(logits, labels)
    return criterion


print("✅ Loss: FocalLoss, BinaryFocalLoss, Hybrid WCE+Focal")
print("   Helpers: mixup, class weights, qwk, accuracy")
print("   Optimizer: AdamW (configured per phase)")
print("   Scheduler: CosineAnnealingLR (configured per phase)")


## Step 18 — Checkpoint & Resume System (Full Recovery)

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# STEP 18 — Checkpoint Save/Load + Resume Infrastructure
# ═══════════════════════════════════════════════════════════════════

def save_checkpoint(path, model, optimizer, scheduler, epoch, phase_idx,
                    val_qwk, img_size, extra=None):
    """Save a full training checkpoint for exact resume."""
    ckpt = {
        "model_state":     model.state_dict(),
        "optimizer_state":  optimizer.state_dict(),
        "scheduler_state":  scheduler.state_dict() if scheduler else None,
        "epoch":            epoch,
        "phase_idx":        phase_idx,
        "val_qwk":          val_qwk,
        "img_size":         img_size,
    }
    if extra:
        ckpt.update(extra)
    torch.save(ckpt, path)


def load_checkpoint(path, model, optimizer=None, scheduler=None, device=DEVICE):
    """Load checkpoint; returns the checkpoint dict."""
    ckpt = safe_load(path, device)
    model.load_state_dict(ckpt["model_state"])
    if optimizer and "optimizer_state" in ckpt:
        try:
            optimizer.load_state_dict(ckpt["optimizer_state"])
        except Exception:
            pass
    if scheduler and ckpt.get("scheduler_state"):
        try:
            scheduler.load_state_dict(ckpt["scheduler_state"])
        except Exception:
            pass
    return ckpt


print("✅ Checkpoint system ready.")
print("   Stores: model, optimizer, scheduler, epoch, phase, QWK, img_size")
print("   Supports exact resume after kernel restart.")


## Step 19 — Training State Management

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# STEP 19 — Training History + State Tracking
# ═══════════════════════════════════════════════════════════════════
# Uses the JSON state system from Step 3 (st_load/st_save/st_get).
# Per-fold and per-phase history is tracked automatically during training.

# Verify state system works
st_save("pipeline_version", "v18")
assert st_get("pipeline_version") == "v18"

print("✅ Training state management ready.")
print("   • JSON-based persistence (survives kernel restarts)")
print("   • Tracks: epoch loss, QWK per fold, best model, training history")
print(f"   • State file: {_STATE_FILE}")


## Step 20 — Training Engine (Single Epoch Loop)

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# STEP 20 — train_one_epoch + validate functions
# ═══════════════════════════════════════════════════════════════════

def train_one_epoch(model, loader, criterion, optimizer,
                    scaler=None, use_mixup=True,
                    grad_accum=1, device=DEVICE):
    model.train()
    total_loss = 0.0
    all_preds, all_labels = [], []
    optimizer.zero_grad()

    for step, batch in enumerate(tqdm(loader, desc="  train", leave=False)):
        imgs, labels = batch[0].to(device), batch[1].to(device)
        do_mixup = use_mixup and labels.dtype == torch.long

        if do_mixup:
            imgs, y_a, y_b, lam = mixup_data(imgs, labels, device=device)

        if scaler is not None:
            with torch.amp.autocast("cuda"):
                logits = model(imgs)
                loss = (mixup_criterion(criterion, logits, y_a, y_b, lam)
                        if do_mixup else criterion(logits, labels))
                loss = loss / grad_accum
            scaler.scale(loss).backward()
            if (step + 1) % grad_accum == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer); scaler.update()
                optimizer.zero_grad()
        else:
            logits = model(imgs)
            loss = (mixup_criterion(criterion, logits, y_a, y_b, lam)
                    if do_mixup else criterion(logits, labels))
            (loss / grad_accum).backward()
            if (step + 1) % grad_accum == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step(); optimizer.zero_grad()

        total_loss += loss.item() * grad_accum
        if labels.dtype == torch.long:
            all_preds.extend(logits.argmax(1).detach().cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

    avg_loss = total_loss / max(len(loader), 1)
    acc = (np.array(all_preds) == np.array(all_labels)).mean() if all_preds else 0.0
    return avg_loss, acc


@torch.no_grad()
def validate(model, loader, criterion=None, device=DEVICE):
    model.eval()
    total_loss = 0.0
    all_probs, all_preds, all_labels = [], [], []

    for imgs, labels in tqdm(loader, desc="  val  ", leave=False):
        imgs, labels = imgs.to(device), labels.to(device)
        logits = model(imgs)
        if criterion is not None:
            total_loss += criterion(logits, labels).item()
        probs = F.softmax(logits, dim=1).cpu().numpy()
        all_probs.extend(probs)
        all_preds.extend(logits.argmax(1).cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

    avg_loss   = total_loss / max(len(loader), 1) if criterion else 0.0
    all_labels = np.array(all_labels)
    all_preds  = np.array(all_preds)
    all_probs  = np.array(all_probs)
    acc        = (all_preds == all_labels).mean()
    kappa      = qwk(all_labels, all_preds) if len(all_labels) > 1 else 0.0
    return avg_loss, acc, kappa, all_probs, all_preds, all_labels


print("✅ Training engine ready: train_one_epoch(), validate()")


## Step 21 — Cross-Validation Training (5-Fold × 3-Phase Execution)

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# STEP 21 — 5-Fold × 3-Phase Training + Early Stopping + OOF
# ═══════════════════════════════════════════════════════════════════
LR           = 3e-4
LR_FINE      = 1e-4
LR_FULL      = 3e-5
WEIGHT_DECAY = 1e-4
USE_MIXUP    = True
GRAD_ACCUM   = 1

PHASES = [
    {"size": 224, "epochs": 15, "name": "P1-Freeze",  "unfreeze": 0,  "lr": LR},
    {"size": 384, "epochs": 40, "name": "P2-Partial", "unfreeze": 4,  "lr": LR_FINE},
    {"size": 512, "epochs": 25, "name": "P3-Full",    "unfreeze": 99, "lr": LR_FULL},
]
BS_MAP       = {224: 32, 384: 16, 512: 8}
ES_PATIENCE  = 5
ES_MIN_DELTA = 0.001

oof_probs  = np.zeros((len(df), NUM_CLASSES), dtype=np.float32)
oof_labels = df["diagnosis"].values.copy()
fold_val_qwks = []

print("=" * 70)
print("  5-FOLD × 3-PHASE TRAINING")
print(f"  Backbone : {BACKBONE}  |  Device : {DEVICE.upper()}")
print(f"  Phases   : {[(p['name'], p['size'], p['epochs']) for p in PHASES]}")
print(f"  ES       : patience={ES_PATIENCE}, min_delta={ES_MIN_DELTA}")
print("=" * 70)

for fold in range(N_FOLDS):
    fold_ckpt = ARTIFACT_DIR / f"fold{fold}_best.pt"
    fold_oof  = ARTIFACT_DIR / f"fold{fold}_oof.npy"
    fold_flag = ARTIFACT_DIR / f"_done_fold{fold}.flag"

    # ── Resume: fold already done ─────────────────────────────────
    if fold_flag.exists() and fold_ckpt.exists():
        if fold_oof.exists():
            val_idx = df[df["fold"] == fold].index
            oof_probs[val_idx] = np.load(str(fold_oof))
        prev_ckpt = safe_load(fold_ckpt, "cpu")
        fold_val_qwks.append(prev_ckpt.get("val_qwk", 0.0))
        print(f"  ✅ [RESUME] Fold {fold} — QWK={fold_val_qwks[-1]:.4f}")
        continue

    print(f"\n  ══ FOLD {fold} ══")
    seed_everything(SEED + fold)

    df_tr_fold = df[df["fold"] != fold].reset_index(drop=True)
    df_va_fold = df[df["fold"] == fold].reset_index(drop=True)
    val_idx    = df[df["fold"] == fold].index

    criterion_fn = build_5class_criterion(df_tr_fold)
    model        = build_5class_model(pretrained=True)

    best_val_qwk = -1.0
    best_state   = None

    for phase_idx, phase in enumerate(PHASES):
        sz, n_ep = phase["size"], phase["epochs"]
        pname    = phase["name"]
        unf      = phase["unfreeze"]
        bs       = BS_MAP[sz]
        p_lr     = phase["lr"]

        # Set freeze state
        if unf == 0:
            model.freeze_backbone()
        elif unf >= 99:
            model.unfreeze_all()
        else:
            model.unfreeze_top_blocks(unf)

        trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"  [{pname}] {sz}px × {n_ep}ep | trainable: {trainable/1e6:.2f}M | bs={bs}")

        tr_ds   = APTOSDataset(df_tr_fold, transform=build_train_transforms(sz), img_size=sz)
        va_ds   = APTOSDataset(df_va_fold, transform=build_val_transforms(sz),   img_size=sz)
        sampler = build_weighted_sampler(df_tr_fold)
        tr_ld   = make_loader(tr_ds, batch_size=bs, sampler=sampler, drop_last=True)
        va_ld   = make_loader(va_ds, batch_size=bs, shuffle=False)

        optimizer = torch.optim.AdamW(
            filter(lambda p: p.requires_grad, model.parameters()),
            lr=p_lr, weight_decay=WEIGHT_DECAY
        )
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_ep, eta_min=1e-6)
        scaler    = torch.amp.GradScaler("cuda") if USE_AMP else None

        es_counter = 0
        for ep in range(n_ep):
            tr_loss, tr_acc = train_one_epoch(
                model, tr_ld, criterion_fn, optimizer,
                scaler=scaler, use_mixup=USE_MIXUP, grad_accum=GRAD_ACCUM
            )
            va_loss, va_acc, va_qwk, va_probs, _, _ = validate(model, va_ld, criterion_fn)
            scheduler.step()

            lr_now = optimizer.param_groups[0]["lr"]
            print(f"    Ep {ep+1:2d}/{n_ep} | TrL={tr_loss:.4f} TrA={tr_acc:.3f} | "
                  f"VaL={va_loss:.4f} VaA={va_acc:.3f} QWK={va_qwk:.4f} | lr={lr_now:.2e}")

            if va_qwk > best_val_qwk + ES_MIN_DELTA:
                best_val_qwk = va_qwk
                best_state   = deepcopy(model.state_dict())
                es_counter   = 0
                save_checkpoint(
                    fold_ckpt, model, optimizer, scheduler,
                    epoch=ep, phase_idx=phase_idx,
                    val_qwk=best_val_qwk, img_size=sz
                )
            else:
                es_counter += 1

            if es_counter >= ES_PATIENCE:
                print(f"    ⏹ Early stopping at epoch {ep+1}")
                break

        # Restore best state before next phase
        if best_state is not None:
            model.load_state_dict(best_state)

        del tr_ds, va_ds, tr_ld, va_ld, optimizer, scheduler, scaler
        gc.collect()
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

    # ── OOF predictions for this fold ─────────────────────────────
    if best_state is not None:
        model.load_state_dict(best_state)
    va_ds_final = APTOSDataset(df_va_fold, transform=build_val_transforms(384), img_size=384)
    va_ld_final = make_loader(va_ds_final, batch_size=16, shuffle=False)
    _, _, _, fold_probs, _, _ = validate(model, va_ld_final)
    oof_probs[val_idx] = fold_probs
    np.save(str(fold_oof), fold_probs)

    fold_val_qwks.append(best_val_qwk)
    fold_flag.touch()
    print(f"  ✅ Fold {fold} done — best QWK={best_val_qwk:.4f}")

    del model; gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

# ── Save OOF ─────────────────────────────────────────────────────
np.save(str(ARTIFACT_DIR / "oof_probs.npy"),  oof_probs)
np.save(str(ARTIFACT_DIR / "oof_labels.npy"), oof_labels)

oof_preds = oof_probs.argmax(1)
oof_qwk   = qwk(oof_labels, oof_preds)
oof_acc   = accuracy(oof_labels, oof_preds)
st_save("oof_qwk", float(oof_qwk))

print("\n" + "=" * 70)
print(f"  OOF QWK      : {oof_qwk:.4f}")
print(f"  OOF Accuracy : {oof_acc*100:.2f}%")
if fold_val_qwks:
    print(f"  Mean fold QWK: {np.mean(fold_val_qwks):.4f} ± {np.std(fold_val_qwks):.4f}")
print("=" * 70)


## Step 22 — Validation Summary (QWK per Fold)

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# STEP 22 — Validation Results Summary
# ═══════════════════════════════════════════════════════════════════
print("Validation QWK per fold:")
for i, q in enumerate(fold_val_qwks):
    marker = " ← best" if i == int(np.argmax(fold_val_qwks)) else ""
    print(f"  Fold {i}: QWK = {q:.4f}{marker}")
print(f"\n  Mean: {np.mean(fold_val_qwks):.4f} ± {np.std(fold_val_qwks):.4f}")
print(f"  OOF QWK (argmax): {st_get('oof_qwk', 'N/A')}")


## Step 23 — Early Stopping (Documentation)Early stopping is integrated into Step 21's training loop.

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# STEP 23 — Early Stopping Configuration (implemented in Step 21)
# ═══════════════════════════════════════════════════════════════════
print("Early Stopping Configuration:")
print(f"  Monitor     : Validation QWK")
print(f"  Patience    : {ES_PATIENCE} epochs")
print(f"  Min Delta   : {ES_MIN_DELTA}")
print(f"  Action      : Restore best weights, move to next phase or end fold")
print("\n  ✅ Already active in the training loop (Step 21).")


## Step 24 — OOF (Out-of-Fold) Predictions

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# STEP 24 — Load & Inspect OOF Predictions
# ═══════════════════════════════════════════════════════════════════
oof_probs_loaded  = np.load(str(ARTIFACT_DIR / "oof_probs.npy"))
oof_labels_loaded = np.load(str(ARTIFACT_DIR / "oof_labels.npy"))

print(f"OOF Predictions loaded:")
print(f"  Shape  : {oof_probs_loaded.shape}")
print(f"  Labels : {oof_labels_loaded.shape}")
print(f"  Argmax QWK      : {qwk(oof_labels_loaded, oof_probs_loaded.argmax(1)):.4f}")
print(f"  Argmax Accuracy  : {accuracy(oof_labels_loaded, oof_probs_loaded.argmax(1))*100:.2f}%")

# Per-class mean probability
print("\n  Mean predicted probabilities per true class:")
for g in range(NUM_CLASSES):
    mask = oof_labels_loaded == g
    if mask.sum() > 0:
        mp = oof_probs_loaded[mask].mean(axis=0)
        print(f"    Grade {g}: {np.round(mp, 3)}")


## Step 25 — Test-Time Augmentation (TTA)

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# STEP 25 — TTA Configuration (used in Steps 27 & 28)
# ═══════════════════════════════════════════════════════════════════
print("TTA (Test-Time Augmentation) Configuration:")
print(f"  Views: {len(tta_transforms_lst)}")
print("    1. Original (resize + normalize)")
print("    2. Horizontal flip")
print("    3. Rotate +10°")
print("    4. Rotate -10°")
print("    5. Vertical flip")
print("  Strategy: Average softmax probabilities across all views.")
print("\n  ✅ TTA transforms already built in Step 14.")


## Step 26 — Threshold Optimisation (on OOF predictions)

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# STEP 26 — Optimise Thresholds using OOF Predictions
# ═══════════════════════════════════════════════════════════════════
from scipy.optimize import minimize

def optimise_thresholds(probs, labels, n_classes=5):
    base_preds = probs.argmax(1)
    best_qwk   = qwk(labels, base_preds)
    best_preds = base_preds.copy()
    print(f"  Argmax QWK (baseline): {best_qwk:.4f}")

    # Phase 1: sweep class-0 boundary bias
    for bias in np.linspace(-0.4, 0.4, 17):
        biased = probs.copy()
        biased[:, 0] += bias
        p = biased.argmax(1)
        q = qwk(labels, p)
        if q > best_qwk:
            best_qwk   = q
            best_preds = p.copy()
            print(f"  Improved QWK={q:.4f} with class-0 bias={bias:.3f}")

    # Phase 2: Nelder-Mead over all class biases
    def neg_qwk_biased(biases):
        b = probs.copy()
        for c, bv in enumerate(biases):
            b[:, c] += bv
        return -qwk(labels, b.argmax(1))

    res = minimize(neg_qwk_biased, x0=np.zeros(n_classes),
                   method="Nelder-Mead",
                   options={"maxiter": 500, "xatol": 1e-4, "fatol": 1e-4})
    opt_biased = probs.copy()
    for c, bv in enumerate(res.x):
        opt_biased[:, c] += bv
    opt_preds = opt_biased.argmax(1)
    opt_q     = qwk(labels, opt_preds)
    if opt_q > best_qwk:
        best_qwk   = opt_q
        best_preds = opt_preds.copy()
        print(f"  Nelder-Mead improved QWK={opt_q:.4f}  biases={np.round(res.x, 3)}")

    return best_preds, best_qwk


print("Threshold optimisation on OOF predictions:")
opt_preds, opt_qwk = optimise_thresholds(oof_probs_loaded, oof_labels_loaded)
print(f"\n  ✅ Optimised OOF QWK: {opt_qwk:.4f}")
print(f"     OOF Accuracy     : {accuracy(oof_labels_loaded, opt_preds)*100:.2f}%")

np.save(str(ARTIFACT_DIR / "oof_opt_preds.npy"), opt_preds)
st_save("opt_qwk", float(opt_qwk))


## Step 27 — Testing on Final Hold-Out Set (Ensemble + TTA)

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# STEP 27 — Ensemble Inference on Hold-Out Test (Fold 0)
# ═══════════════════════════════════════════════════════════════════
TEST_FOLD  = 0
df_test    = df[df["fold"] == TEST_FOLD].reset_index(drop=True)
print(f"Ensemble inference on fold {TEST_FOLD} ({len(df_test):,} samples) using folds 1-4 ...")

ensemble_probs = np.zeros((len(df_test), NUM_CLASSES), dtype=np.float32)
n_models       = 0

for fold in range(1, N_FOLDS):
    fold_ckpt = ARTIFACT_DIR / f"fold{fold}_best.pt"
    if not fold_ckpt.exists():
        print(f"  ⚠️  fold{fold}_best.pt not found — skipping")
        continue

    ckpt   = safe_load(fold_ckpt, DEVICE)
    model  = build_5class_model(pretrained=False)
    model.load_state_dict(ckpt["model_state"])
    model.eval()
    img_sz = ckpt.get("img_size", 384)

    fold_probs = np.zeros((len(df_test), NUM_CLASSES), dtype=np.float32)
    with torch.no_grad():
        for tta_tf in build_tta_transforms(img_sz):
            ds  = APTOSDataset(df_test, transform=tta_tf, img_size=img_sz)
            ld  = make_loader(ds, batch_size=BS_MAP.get(img_sz, 16), shuffle=False)
            bps = []
            for imgs, _ in tqdm(ld, desc=f"  fold{fold} TTA", leave=False):
                logits = model(imgs.to(DEVICE))
                bps.append(F.softmax(logits, 1).cpu().numpy())
            fold_probs += np.concatenate(bps, axis=0)

    fold_probs     /= len(tta_transforms_lst)
    ensemble_probs += fold_probs
    n_models       += 1

    fold_q = qwk(df_test["diagnosis"].values, fold_probs.argmax(1))
    print(f"  Fold {fold}: QWK={fold_q:.4f}")
    del model; gc.collect()

if n_models > 0:
    ensemble_probs /= n_models
    test_preds  = ensemble_probs.argmax(1)
    test_labels = df_test["diagnosis"].values
    test_qwk_v  = qwk(test_labels, test_preds)
    test_acc_v  = accuracy(test_labels, test_preds)

    print(f"\n  ✅ Ensemble ({n_models} models × {len(tta_transforms_lst)} TTA views)")
    print(f"     Test QWK      : {test_qwk_v:.4f}")
    print(f"     Test Accuracy : {test_acc_v*100:.2f}%")

    np.save(str(ARTIFACT_DIR / "ensemble_probs.npy"), ensemble_probs)
    np.save(str(ARTIFACT_DIR / "test_preds.npy"),     test_preds)
    np.save(str(ARTIFACT_DIR / "test_labels.npy"),    test_labels)
    st_save("test_qwk", float(test_qwk_v))
    st_save("test_acc", float(test_acc_v))
else:
    print("⚠️  No fold checkpoints found. Run Step 21 first.")


## Step 28 — Metrics & Evaluation (Confusion Matrix, Per-Class Recall)

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# STEP 28 — OOF + Test Metrics Visualisation
# ═══════════════════════════════════════════════════════════════════
%matplotlib inline

_cm_path = PLOT_DIR / "oof_confusion_matrix.png"

oof_probs_l  = np.load(str(ARTIFACT_DIR / "oof_probs.npy"))
oof_labels_l = np.load(str(ARTIFACT_DIR / "oof_labels.npy"))
oof_pred_l   = oof_probs_l.argmax(1)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

cm   = confusion_matrix(oof_labels_l, oof_pred_l)
disp = ConfusionMatrixDisplay(cm, display_labels=[f"G{i}" for i in range(5)])
disp.plot(ax=axes[0], colorbar=False, cmap="Blues")
axes[0].set_title(
    f"OOF Confusion Matrix\nQWK={qwk(oof_labels_l, oof_pred_l):.4f}  "
    f"Acc={accuracy(oof_labels_l, oof_pred_l)*100:.1f}%",
    fontweight="bold")

per_class_recall = cm.diagonal() / np.maximum(cm.sum(axis=1), 1)
axes[1].bar([GRADE_MAP[i] for i in range(5)], per_class_recall,
            color=GRADE_COLORS, edgecolor="k")
axes[1].axhline(y=0.85, color="red", linestyle="--", label="Target 85%")
axes[1].set_ylim(0, 1.05)
axes[1].set_ylabel("Recall")
axes[1].set_title("Per-Class Recall (OOF)", fontweight="bold")
axes[1].tick_params(axis="x", rotation=20)
axes[1].legend()
for i, v in enumerate(per_class_recall):
    axes[1].text(i, v + 0.01, f"{v:.2f}", ha="center", fontsize=9)

plt.tight_layout()
plt.savefig(_cm_path, dpi=150, bbox_inches="tight")
plt.show()

print(f"\nClassification Report (OOF):")
print(classification_report(oof_labels_l, oof_pred_l,
      target_names=[GRADE_MAP[i] for i in range(5)]))


## Step 29 — Model Export

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# STEP 29 — Export Final Model + Thresholds + Label Mapping
# ═══════════════════════════════════════════════════════════════════
import shutil as _sh

EXPORT_DIR  = ARTIFACT_DIR / "export_v18"
EXPORT_FLAG = ARTIFACT_DIR / "_done_export.flag"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

if EXPORT_FLAG.exists():
    print("✅ [RESUME] Model export already done.")
else:
    _bf  = int(np.argmax(fold_val_qwks)) if fold_val_qwks else 0
    _src = ARTIFACT_DIR / f"fold{_bf}_best.pt"
    if _src.exists():
        _sh.copy2(_src, EXPORT_DIR / "best_model.pt")
        print(f"best_model.pt saved (fold {_bf})")
    else:
        print(f"⚠️  fold{_bf}_best.pt not found — train K-Fold first.")

    # Thresholds, label map, metadata
    thr_data  = {"stage1_threshold": st_get("stage1_threshold", 0.5),
                 "oof_qwk": st_get("oof_qwk"), "opt_qwk": st_get("opt_qwk")}
    label_map = {"grade_map": GRADE_MAP, "num_classes": NUM_CLASSES,
                 "backbone": BACKBONE, "img_size": 384}
    meta      = {"fold_qwks": fold_val_qwks if fold_val_qwks else [],
                 "oof_qwk": st_get("oof_qwk"), "opt_qwk": st_get("opt_qwk"),
                 "test_qwk": st_get("test_qwk"), "device": DEVICE}

    (EXPORT_DIR / "thresholds.json").write_text(json.dumps(thr_data, indent=2))
    (EXPORT_DIR / "label_map.json").write_text(json.dumps(label_map, indent=2))
    (EXPORT_DIR / "metadata.json").write_text(json.dumps(meta, indent=2))

    EXPORT_FLAG.touch()
    print(f"\n✅ Export complete → {EXPORT_DIR}")
    for f in sorted(EXPORT_DIR.rglob("*")):
        if f.is_file():
            print(f"  {f.name}  ({f.stat().st_size / 1e3:.1f} KB)")


## Step 30 — Grad-CAM++ Explainability + Deployment Info

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# STEP 30 — Grad-CAM++ Explainability + Deployment Summary
# ═══════════════════════════════════════════════════════════════════
%matplotlib inline

try:
    from pytorch_grad_cam import GradCAMPlusPlus
    from pytorch_grad_cam.utils.image import show_cam_on_image
    from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
    _GRADCAM_OK = True
except ImportError:
    print("⚠️  grad-cam not installed. Run: pip install grad-cam")
    _GRADCAM_OK = False

if _GRADCAM_OK and fold_val_qwks:
    _gcam_out = PLOT_DIR / "gradcam_board_v18.png"

    best_fold  = int(np.argmax(fold_val_qwks))
    ckpt       = safe_load(ARTIFACT_DIR / f"fold{best_fold}_best.pt", "cpu")
    gcam_model = build_5class_model(pretrained=False)
    gcam_model.load_state_dict(ckpt["model_state"])
    gcam_model.eval()
    gcam_model.to("cpu")

    target_layers = [gcam_model.backbone.blocks[-1][-1]]
    cam = GradCAMPlusPlus(model=gcam_model, target_layers=target_layers)

    n_per_class = 2
    fig, axes   = plt.subplots(NUM_CLASSES, n_per_class * 2,
                               figsize=(14, NUM_CLASSES * 3))

    for grade in range(NUM_CLASSES):
        samples = df[df["diagnosis"] == grade].sample(
            min(n_per_class, (df["diagnosis"] == grade).sum()),
            random_state=SEED)

        for j, (_, row) in enumerate(samples.iterrows()):
            img_np    = preprocess_fundus(row["image_path"], size=384)
            tf        = build_val_transforms(384)
            inp       = tf(image=img_np)["image"].unsqueeze(0)
            rgb_float = img_np.astype(np.float32) / 255.0

            with torch.no_grad():
                pred_class = gcam_model(inp).argmax(1).item()

            grayscale = cam(input_tensor=inp,
                            targets=[ClassifierOutputTarget(grade)])
            cam_img   = show_cam_on_image(rgb_float, grayscale[0], use_rgb=True)

            col_orig = j * 2
            col_cam  = j * 2 + 1
            axes[grade][col_orig].imshow(img_np)
            axes[grade][col_orig].axis("off")
            if j == 0:
                axes[grade][col_orig].set_ylabel(
                    f"G{grade}\n{GRADE_MAP[grade]}",
                    fontsize=9, fontweight="bold", color=GRADE_COLORS[grade],
                    rotation=0, labelpad=60, va="center")
            axes[grade][col_cam].imshow(cam_img)
            axes[grade][col_cam].axis("off")
            axes[grade][col_cam].set_title(
                f"Pred: G{pred_class}", fontsize=8,
                color="green" if pred_class == grade else "red")

    plt.suptitle(
        f"Grad-CAM++ — Best Fold {best_fold} (QWK={fold_val_qwks[best_fold]:.4f})",
        fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig(_gcam_out, dpi=130, bbox_inches="tight")
    plt.show()
    print(f"✅ Grad-CAM++ board saved → {_gcam_out}")
    del gcam_model; gc.collect()
elif _GRADCAM_OK:
    print("⚠️  fold_val_qwks not available — run training first.")

# ── Deployment Summary ────────────────────────────────────────────
print("\n" + "=" * 68)
print("  DEPLOYMENT OPTIONS")
print("=" * 68)
print("  1. Streamlit: streamlit run app.py")
print("     → Upload retinal image → Get DR grade + Grad-CAM heatmap")
print("  2. Hugging Face Spaces: push export folder to HF repo")
print("  3. API: wrap model in FastAPI / Flask")
print(f"\n  Exported artifacts: {EXPORT_DIR}")

# ── Final Results ─────────────────────────────────────────────────
state = st_load()
print("\n" + "=" * 68)
print("  FINAL RESULTS SUMMARY")
print("=" * 68)
print(f"  Backbone       : {BACKBONE}")
print(f"  Device         : {DEVICE.upper()}")
print(f"  Dataset        : APTOS 2019 ({len(df):,} images)")
if fold_val_qwks:
    for i, q in enumerate(fold_val_qwks):
        print(f"    Fold {i}: QWK={q:.4f}")
    print(f"    Mean QWK     : {np.mean(fold_val_qwks):.4f} ± {np.std(fold_val_qwks):.4f}")
print(f"  OOF QWK        : {state.get('oof_qwk', 'N/A')}")
print(f"  Optimised QWK  : {state.get('opt_qwk', 'N/A')}")
tq = state.get("test_qwk")
ta = state.get("test_acc")
print(f"  Test QWK       : {tq if tq else 'N/A'}")
print(f"  Test Accuracy  : {float(ta)*100:.2f}%" if ta else "  Test Accuracy  : N/A")
print("=" * 68)
print("  ⚠️  RESEARCH USE ONLY — NOT FOR CLINICAL DEPLOYMENT")
print("=" * 68)
